# Avaliação do Modelo MoE-DQN para SOR

Este notebook compara o desempenho da estratégia baseline **TWAP** (Time-Weighted Average Price) com o agente inteligente **MoE-DQN** (Mixture of Experts - Deep Q-Network) na execução de ordens entre as venues B3 e Base.

**Objetivos:**
- Avaliar a qualidade de execução através do preço médio e slippage
- Medir a economia gerada pelo roteamento inteligente versus roteamento cego
- Calcular métricas como VSOT (VWAP Since Order Time) e Implementation Shortfall
- Visualizar a superioridade do modelo treinado em cenários out-of-sample

**Resultados principais:**
- Preço médio TWAP: R$ {avg_twap:.4f}
- Preço médio MoE-DQN: R$ {avg_ia:.4f}
- Economia total gerada: R$ {economia_total:.2f} para ordem de {TOTAL_ORDER} ações
- Taxa de rejeição TWAP: {stats_twap['rejects']} | Taxa de rejeição MoE-DQN: {stats_ia['rejects']}

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def df_to_l2_numpy(df: pd.DataFrame, levels: int = 5) -> dict:
    """
    Converte DataFrame L2 (ask_1..5, vol_ask_1..5) em dict de arrays float32.
    Mantém o shape e reduz overhead no step().
    """
    out = {}
    for i in range(1, levels + 1):
        out[f"ask_{i}"] = df[f"ask_{i}"].to_numpy(dtype=np.float32, copy=False)
        out[f"vol_ask_{i}"] = df[f"vol_ask_{i}"].to_numpy(dtype=np.float32, copy=False)
    return out

In [ ]:
# Configuração de caminho para importar o pacote local `src`
import sys
from pathlib import Path

ROOT_DIR = Path('..').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Importando as classes e funções de avaliação
from src.sor_env_numpy import MultiVenueSOREnvNumpy
from src.moe_dqn import MoENetwork
from src.evaluate_baselines import simulate_twap, evaluate_agent

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

def assert_env_info_keys(info: dict) -> None:
    required = {
        "inventory_left", "arrival_price",
        "executed_cost", "executed_volume",
        "avg_price", "slippage", "is_valid", "rejection_reason",
        "t", "T",
    }
    missing = required - set(info.keys())
    if missing:
        raise RuntimeError(f"Env.step() não está retornando info auditável. Faltam: {missing}")

### Geração de Dados de Teste (Out of Sample)

In [ ]:
steps = 500
np.random.seed(99) # Semente diferente da usada no treinamento

df_b3 = pd.DataFrame({
    'ask_1': np.random.uniform(47.05, 47.15, steps), 'vol_ask_1': np.random.randint(100, 500, steps),
    'ask_2': np.random.uniform(47.16, 47.25, steps), 'vol_ask_2': np.random.randint(100, 500, steps),
    'ask_3': np.random.uniform(47.26, 47.35, steps), 'vol_ask_3': np.random.randint(100, 500, steps),
    'ask_4': np.random.uniform(47.36, 47.45, steps), 'vol_ask_4': np.random.randint(100, 500, steps),
    'ask_5': np.random.uniform(47.46, 47.55, steps), 'vol_ask_5': np.random.randint(100, 500, steps),
})

df_base = pd.DataFrame({
    'ask_1': np.random.uniform(47.00, 47.10, steps), 'vol_ask_1': np.random.randint(50, 300, steps),
    'ask_2': np.random.uniform(47.11, 47.20, steps), 'vol_ask_2': np.random.randint(50, 300, steps),
    'ask_3': np.random.uniform(47.21, 47.30, steps), 'vol_ask_3': np.random.randint(50, 300, steps),
    'ask_4': np.random.uniform(47.31, 47.40, steps), 'vol_ask_4': np.random.randint(50, 300, steps),
    'ask_5': np.random.uniform(47.41, 47.50, steps), 'vol_ask_5': np.random.randint(50, 300, steps),
})

# --- Simulando "market_vol" (proxy) para VSOT ---
vol_cols = [f"vol_ask_{i}" for i in range(1, 6)]
df_b3["market_vol"] = df_b3[vol_cols].sum(axis=1).astype(float)

# opcional: evitar volume zero por segurança
df_b3["market_vol"] = df_b3["market_vol"].clip(lower=1.0)

print("Dados de teste gerados com sucesso.")

### Carregando o Modelo e Executando a Comparação

In [ ]:
TOTAL_ORDER = 10000

lob_b3_np = df_to_l2_numpy(df_b3)
lob_base_np = df_to_l2_numpy(df_base)

env_twap = MultiVenueSOREnvNumpy(lob_b3=lob_b3_np, lob_base=lob_base_np, total_inventory=TOTAL_ORDER)
env_ia   = MultiVenueSOREnvNumpy(lob_b3=lob_b3_np, lob_base=lob_base_np, total_inventory=TOTAL_ORDER)

caminho_modelo = ROOT_DIR / 'models' / 'moe_dqn_sor.pth'
agente_treinado = MoENetwork(input_dim=5, output_dim=4, num_experts=3)
agente_treinado.load_state_dict(torch.load(caminho_modelo, map_location="cpu"))
agente_treinado.eval()

preco_twap, vol_twap, stats_twap = simulate_twap(env_twap, TOTAL_ORDER)
preco_ia, vol_ia, stats_ia = evaluate_agent(agente_treinado, env_ia)

print("\n--- RESULTADOS (AUDITÁVEIS) ---")
print(f"Arrival Price: R$ {arrival:.4f}")
print(f"TWAP -> avg_price: R$ {avg_twap:.4f} | IS/share: R$ {avg_twap - arrival:.6f} | rejects: {stats_twap['rejects']} | steps: {stats_twap['steps']}")
print(f"IA   -> avg_price: R$ {avg_ia:.4f} | IS/share: R$ {avg_ia - arrival:.6f} | rejects: {stats_ia['rejects']} | steps: {stats_ia['steps']}")

economia_total = (avg_twap - avg_ia) * TOTAL_ORDER
print(f"Economia total (R$): {economia_total:.2f}")

In [ ]:
def vsot_proxy_from_l2(df_b3: pd.DataFrame, steps: int) -> float:
    """
    Proxy de VSOT usando L2:
    price_t = VWAP(ask_1..5, vol_ask_1..5) por step
    e depois média ponderada no tempo pelos volumes do próprio L2.
    """
    s = int(max(1, min(steps, len(df_b3))))
    prices = []
    vols = []
    for t in range(s):
        px = np.array([df_b3.iloc[t][f"ask_{i}"] for i in range(1,6)], dtype=float)
        vv = np.array([df_b3.iloc[t][f"vol_ask_{i}"] for i in range(1,6)], dtype=float)
        denom = vv.sum()
        step_vwap = (px * vv).sum() / denom if denom > 0 else px.mean()
        prices.append(step_vwap)
        vols.append(denom)

    prices = np.array(prices, dtype=float)
    vols = np.array(vols, dtype=float)
    return float((prices * vols).sum() / vols.sum()) if vols.sum() > 0 else float(prices.mean())

vsot_twap = vsot_proxy_from_l2(df_b3, stats_twap["steps"])
vsot_ia   = vsot_proxy_from_l2(df_b3, stats_ia["steps"])

print("\n--- VSOT (proxy L2) ---")
print(f"VSOT TWAP: R$ {vsot_twap:.4f}")
print(f"VSOT IA  : R$ {vsot_ia:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import torch

def run_and_log_twap(env):
    """TWAP discreto: sempre action=1 (B3). Loga info por step."""
    rows = []
    state, _ = env.reset()
    done = False
    step_idx = 0

    while not done:
        action = 1
        state, reward, terminated, truncated, info = env.step(action)
        done = bool(terminated or truncated)

        rows.append({
            "step": step_idx,
            "action": action,
            "reward": float(reward),
            **{k: info.get(k, None) for k in [
                "inventory_left","arrival_price","executed_volume","executed_cost",
                "avg_price","slippage","is_valid","rejection_reason","t","T"
            ]}
        })
        step_idx += 1

    return pd.DataFrame(rows)

def run_and_log_agent(env, model):
    """IA: argmax(Q). Loga info por step e também gate weights."""
    rows = []
    state, _ = env.reset()
    done = False
    step_idx = 0

    while not done:
        st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q = model(st)
            action = int(torch.argmax(q, dim=1).item())
            # gate weights (MoE)
            gate = torch.softmax(model.gating_network(st), dim=-1).squeeze(0).cpu().numpy()

        state, reward, terminated, truncated, info = env.step(action)
        done = bool(terminated or truncated)

        row = {
            "step": step_idx,
            "action": action,
            "reward": float(reward),
            "gate_0": float(gate[0]),
            "gate_1": float(gate[1]),
            "gate_2": float(gate[2]),
            **{k: info.get(k, None) for k in [
                "inventory_left","arrival_price","executed_volume","executed_cost",
                "avg_price","slippage","is_valid","rejection_reason","t","T"
            ]}
        }
        rows.append(row)
        step_idx += 1

    return pd.DataFrame(rows)

def summarize_log(df):
    total_cost = float(df["executed_cost"].fillna(0).sum())
    total_vol  = float(df["executed_volume"].fillna(0).sum())
    avg_price = (total_cost / total_vol) if total_vol > 0 else 0.0
    rejects = int((~df["is_valid"].astype(bool)).sum()) if "is_valid" in df else 0
    steps = int(df.shape[0])
    arrival = float(df["arrival_price"].dropna().iloc[0]) if df["arrival_price"].notna().any() else np.nan
    return {
        "avg_price": avg_price,
        "total_cost": total_cost,
        "total_vol": total_vol,
        "rejects": rejects,
        "steps": steps,
        "arrival_price": arrival,
        "fill_rate": (total_vol / float(env.total_inventory)) if hasattr(env, "total_inventory") else np.nan
    }

In [ ]:
log_twap = run_and_log_twap(env_twap)
log_ia   = run_and_log_agent(env_ia, agente_treinado)

env = type("EnvProxy", (), {"total_inventory": TOTAL_ORDER})()
stats_twap = summarize_log(log_twap)
stats_ia   = summarize_log(log_ia)

stats_twap, stats_ia

In [ ]:
def calcular_vsot(df_b3, steps_taken, price_col="ask_1", vol_col="market_vol") -> float:
    steps_taken = int(max(1, min(steps_taken, len(df_b3))))  # clamp
    precos = df_b3[price_col].iloc[:steps_taken].astype(float)
    vols = df_b3[vol_col].iloc[:steps_taken].astype(float)

    denom = float(vols.sum())
    if denom <= 0:
        return float(precos.mean())
    return float((precos * vols).sum() / denom)

steps_twap = stats_twap["steps"]
steps_ia = stats_ia["steps"]

vsot_twap = calcular_vsot(df_b3, steps_twap)
vsot_ia   = calcular_vsot(df_b3, steps_ia)

arrival = float(stats_ia["arrival_price"])  # mesmo benchmark do env
avg_twap = float(stats_twap["avg_price"])
avg_ia = float(stats_ia["avg_price"])

print("\n--- VSOT (VWAP Since Order Time) ---")
print(f"VSOT TWAP (até step {steps_twap}): R$ {vsot_twap:.4f}")
print(f"VSOT IA   (até step {steps_ia}): R$ {vsot_ia:.4f}")

print("\n--- Implementation Shortfall (vs Arrival) ---")
print(f"Arrival Price: R$ {arrival:.4f}")
print(f"TWAP -> IS/share: R$ {avg_twap - arrival:.6f}")
print(f"IA   -> IS/share: R$ {avg_ia - arrival:.6f}")

economia_total = (avg_twap - avg_ia) * TOTAL_ORDER
print(f"\nEconomia total (R$): {economia_total:.2f}")

In [ ]:
# Proxy de volume de mercado 
vol_cols = [f"vol_ask_{i}" for i in range(1, 6)]
df_b3["market_vol"] = df_b3[vol_cols].sum(axis=1).astype(float).clip(lower=1.0)

def running_vsot(df, steps, price_col="ask_1", vol_col="market_vol"):
    steps = int(min(max(1, steps), len(df)))
    p = df[price_col].iloc[:steps].astype(float).to_numpy()
    v = df[vol_col].iloc[:steps].astype(float).to_numpy()
    cum_pv = np.cumsum(p * v)
    cum_v  = np.cumsum(v)
    return cum_pv / np.maximum(cum_v, 1e-12)

vsot_curve_twap = running_vsot(df_b3, stats_twap["steps"])
vsot_curve_ia   = running_vsot(df_b3, stats_ia["steps"])

vsot_twap = float(vsot_curve_twap[-1])
vsot_ia   = float(vsot_curve_ia[-1])

### Gráficos Analiticos

Nesta seção, apresentamos uma análise visual comparativa entre as estratégias de execução. Os gráficos a seguir ilustram:

1. **Comparação de Preço Médio**: Contrasta o preço médio de execução entre TWAP (baseline) e MoE-DQN, destacando o melhor desempenho de preço da IA.

2. **Economia Gerada**: Quantifica o ganho financeiro absoluto (em R$) obtido pelo roteamento inteligente versus o roteamento cego, considerando o volume total da ordem.

3. **Inventário ao Longo do Tempo**: Mostra a progressão de execução (redução de inventário) para ambas as estratégias, revelando diferenças na agressividade e urgência.

4. **Volume Executado por Step**: Detalha o comportamento microestrutural de preenchimento de ordens, evidenciando padrões de execução distintos.

5. **Slippage com Rejeições**: Monitora o slippage (diferença entre preço médio e arrival price) ao longo do tempo, marcando rejeições que impactam a execução.

6. **Pesos do Gating Network (MoE)**: Visualiza como o mecanismo de seleção de especialistas alterna entre os três experts, demonstrando especialização dinâmica.

7. **Distribuição de Ações**: Compara a frequência de cada ação (Wait, B3, Base, Slice) entre as duas estratégias.

8. **Curva de VSOT**: Acompanha o benchmark de mercado (VWAP Since Order Time) durante o horizonte de execução, comparando a evolução para ambas as estratégias.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Comparação de Preço Médio (Slippage)
modelos = ['TWAP (Baseline)', 'MoE-DQN (IA)']
precos = [preco_twap, preco_ia]
ymin, ymax = min(precos), max(precos)
span = max(ymax - ymin, 1e-6)

ax1.set_ylim(ymin - 0.2*span, ymax + 0.4*span)

cores = ['#e74c3c', '#2ecc71']

bars = ax1.bar(modelos, precos, color=cores, width=0.5)
ax1.set_title('Comparação de Preço Médio de Execução\n(Menor é Melhor)', fontsize=14, pad=15)
ax1.set_ylabel('Preço Médio (R$)', fontsize=12)
ax1.set_ylim(min(precos) * 0.999, max(precos) * 1.001)

for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.001, f'R$ {yval:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Gráfico 2: Economia Total Gerada
economia_por_acao = preco_twap - preco_ia
economia_total = economia_por_acao * TOTAL_ORDER

ax2.bar(['Economia Total'], [economia_total], color='#3498db', width=0.3)
ax2.set_title(f'Economia Gerada pelo SOR Inteligente\n(Ordem de {TOTAL_ORDER} ações)', fontsize=14, pad=15)
ax2.set_ylabel('Valor Economizado (R$)', fontsize=12)
ax2.text(0, economia_total + (abs(economia_total)*0.05), f'R$ {economia_total:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.1 Inventário ao longo do tempo (urgência / agressividade)



In [ ]:
plt.figure(figsize=(12,4))
plt.plot(log_twap["step"], log_twap["inventory_left"], label="TWAP inventory", linewidth=2)
plt.plot(log_ia["step"], log_ia["inventory_left"], label="IA inventory", linewidth=2)
plt.title("Inventário remanescente por step (menor = mais execução)")
plt.xlabel("Step")
plt.ylabel("Inventory left")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()

### 3.2 Volume executado por step (fill microestrutural)



In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    log_twap["step"], log_twap["executed_volume"],
    label="TWAP executed vol", alpha=0.7, linewidth=1.5,
    marker="o", markersize=3
)
ax.plot(
    log_ia["step"], log_ia["executed_volume"],
    label="IA executed vol", alpha=0.7, linewidth=1.5,
    marker="s", markersize=3
)

ax.set_title("Volume executado por step")
ax.set_xlabel("Step")
ax.set_ylabel("Executed volume")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 Slippage por step (com rejeições destacadas)



In [ ]:
plt.figure(figsize=(12,4))
plt.plot(log_twap["step"], log_twap["slippage"], label="TWAP slippage", linewidth=2)
plt.plot(log_ia["step"], log_ia["slippage"], label="IA slippage", linewidth=2)

rej_idx = log_ia.index[~log_ia["is_valid"].astype(bool)]
if len(rej_idx) > 0:
    plt.scatter(log_ia.loc[rej_idx, "step"], log_ia.loc[rej_idx, "slippage"], color="red", label="IA rejects", zorder=3)

plt.title("Slippage por step (R$) + rejeições")
plt.xlabel("Step")
plt.ylabel("Slippage (avg_price - arrival)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()

### 3.4 Gate do MoE ao longo do tempo (prova de especialização/alternância)



In [ ]:
plt.figure(figsize=(12,4))
plt.plot(log_ia["step"], log_ia["gate_0"], label="Expert 0", marker="o", linewidth=2)
plt.plot(log_ia["step"], log_ia["gate_1"], label="Expert 1", marker="s", linewidth=2)
plt.plot(log_ia["step"], log_ia["gate_2"], label="Expert 2", marker="^", linewidth=2)
plt.title("MoE gating weights por step")
plt.xlabel("Step")
plt.ylabel("Gate weight (softmax)")
plt.ylim(0,1)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()

### 3.5 Distribuição das ações (IA vs TWAP)



In [ ]:
from collections import Counter

def plot_action_dist(actions, title):
    counts = Counter(actions)
    keys = [0,1,2,3]
    vals = [counts.get(k,0) for k in keys]
    labels = ["Wait(0)","B3(1)","Base(2)","Slice(3)"]
    plt.bar(labels, vals)
    plt.title(title)
    plt.ylabel("Count")
    plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plot_action_dist(log_twap["action"].tolist(), "TWAP: distribuição de ações")
plt.subplot(1,2,2)
plot_action_dist(log_ia["action"].tolist(), "IA: distribuição de ações")
plt.tight_layout()
plt.show()

### 3.6 Curva de VSOT (benchmark de mercado “desde o início”)



In [ ]:
plt.figure(figsize=(12,4))
plt.plot(vsot_curve_twap, label=f"VSOT curve (TWAP) final={vsot_twap:.4f}", linewidth=2)
plt.plot(vsot_curve_ia,   label=f"VSOT curve (IA) final={vsot_ia:.4f}", linewidth=2)
plt.title("VSOT (VWAP Since Order Time) ao longo do tempo")
plt.xlabel("Step")
plt.ylabel("VSOT (R$)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()